# Phase 3.3：生成实验报告与优化边界

## 目标

读取前两课的实验记录，把性能、质量、环境和限制写成一份可复现报告。ONNX/INT8 在本课只做能力检查，不伪造未执行的加速结果。

**本课交付：** `docs/phase3_baseline_report.md`。

## Evidence Quest 任务卡：Phase 3.3：证据实验报告厅

**你的身份：** 技术调查记者  
**案件背景：** 董事会只看结论，但工程团队需要知道数字的适用范围、失败原因和不能声称的事情。

### 本关专业 Goal

把实验数据写成一页能被真实观众审阅的性能与质量报告。

### 你要交付的作品

**Phase 3 实验报告 + 一条简历级结论**

### 通关判定

- 先运行带逐行中文注释的示范，预测输出，再自己重新敲一遍关键代码。
- 至少改变一个参数或输入，记录它为什么改变了结果。
- 完成末尾的 Boss Challenge，并能解释一个失败样本。
- 把本关产物交给下一关，而不是把代码停留在 Notebook 屏幕上。

**通关奖励：** 解锁徽章：证据报告员  
**学习节奏：** 看故事 -> 跟敲一小段 -> 观察输出 -> 自己改写 -> 验收作品。

## 1. 报告为什么要包含限制？

专业报告不仅展示最好数字，还要写清楚数据规模、硬件、迭代次数、模型路径和不能推广的地方。当前项目使用小型本地语料，报告应明确这是教学 baseline，不是线上 SLA。

In [1]:
# 导入 Path，用它表示跨平台的文件路径。
from pathlib import Path

# 导入 json，用它读取和保存项目的结构化数据。
import json

# 导入 sys，用它把项目根目录加入 Python 的模块搜索路径。
import sys


# 定义一个函数，负责从当前工作目录向上查找项目根目录。
def find_project_root() -> Path:
    # 把当前目录和它的所有父目录放进候选列表。
    candidates = [Path.cwd(), *Path.cwd().parents]

    # 逐个检查候选目录是否包含本项目的两个核心模块目录。
    for candidate in candidates:
        # 找到同时存在的目录时，返回这个候选目录。
        if (candidate / "phase1_doc_parser").is_dir() and (candidate / "phase2_semantic_search").is_dir():
            return candidate

    # 如果所有候选目录都不符合，说明 Jupyter 启动位置不在项目内。
    raise RuntimeError("找不到项目根目录，请从 ai-search-rag-internship 启动 JupyterLab")


# 执行查找函数，得到当前项目根目录。
ROOT = find_project_root()

# 如果项目根目录还不在模块搜索路径中，就把它添加进去。
if str(ROOT) not in sys.path:
    # 把项目根目录插入最前面，确保导入的是当前项目代码。
    sys.path.insert(0, str(ROOT))

# 打印根目录，帮助学习者确认 Notebook 没有在错误目录运行。
print("项目根目录:", ROOT)

项目根目录: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship


In [2]:
# 定义本关任务编号，后面的记录会用它区分不同阶段。
QUEST_STAGE = 'phase3.3'

# 定义学习者可以持续保存的案件档案路径。
QUEST_PROFILE_PATH = ROOT / "data" / "processed" / "evidence_quest_profile.json"

# 如果第一次打开课程还没有档案，就使用一个安全的默认案件。
default_profile = {
    "case_name": "校园知识库失踪案",
    "audience": "需要快速查证资料的同学",
    "must_answer": "证据来自哪里，能否回到原文？",
    "must_refuse": "检索结果没有证据时必须说不知道",
    "xp": 0,
    "badges": [],
}

# 检查任务档案是否已经由 Mission Control 创建。
if QUEST_PROFILE_PATH.is_file():
    # 读取学员自己的案件主题，让所有 Notebook 共享同一个故事。
    quest_profile = json.loads(QUEST_PROFILE_PATH.read_text(encoding="utf-8"))
else:
    # 没有档案时复制默认值，避免直接修改模板字典。
    quest_profile = dict(default_profile)

# 计算当前累计经验值；错误值按 0 处理，避免看板阻塞学习。
quest_xp = int(quest_profile.get("xp", 0))

# 读取已经获得的徽章，并复制成当前 Notebook 的列表。
quest_badges = list(quest_profile.get("badges", []))

# 用可见的文字看板告诉学习者自己正在解决哪个真实问题。
print("Evidence Quest / 当前关卡:", QUEST_STAGE)
print("案件:", quest_profile.get("case_name", default_profile["case_name"]))
print("服务对象:", quest_profile.get("audience", default_profile["audience"]))
print("累计 XP:", quest_xp, "| 徽章:", ", ".join(quest_badges) if quest_badges else "尚未获得")

Evidence Quest / 当前关卡: phase3.3
案件: 校园知识库失踪案
服务对象: 需要复习课程资料的同学
累计 XP: 0 | 徽章: 案件接收员


In [3]:
# 读取 Phase 3.1 的计时基线。
timing_path = ROOT / "data" / "processed" / "phase3_timing_baseline.json"
timing_record = json.loads(timing_path.read_text(encoding="utf-8"))

# 读取 Phase 3.2 的单变量实验。
experiment_path = ROOT / "data" / "processed" / "phase3_single_variable_experiments.json"
experiment_record = json.loads(experiment_path.read_text(encoding="utf-8"))

# 打印关键基线，确认输入文件确实存在并可读取。
print("timing baseline:", timing_record)
print("top_k experiments:", len(experiment_record["top_k_results"]))
print("chunk experiments:", len(experiment_record["chunk_results"]))

# 至少要有一组结果才能生成报告。
assert experiment_record["top_k_results"]
assert experiment_record["chunk_results"]

timing baseline: {'query': 'Chunk overlap', 'top_k': 5, 'warmup': 5, 'iterations': 30, 'python': '3.13.5', 'index_version': 'chunks-4-size-128-overlap-32', 'first_ms': 0.12040000001434237, 'mean_ms': 0.031226666275567066, 'p50_ms': 0.02925000444520265, 'p95_ms': 0.04599998646881431}
top_k experiments: 3
chunk experiments: 3


In [4]:
# 导入 importlib.util 检查可选优化依赖。
import importlib.util

# 记录 ONNX 运行时是否安装。
onnx_available = importlib.util.find_spec("onnxruntime") is not None

# 记录 psutil 是否安装，用于以后测内存。
psutil_available = importlib.util.find_spec("psutil") is not None

# 输出依赖能力，但不因缺少可选包而伪造实验结果。
print({"onnxruntime": onnx_available, "psutil": psutil_available})

{'onnxruntime': True, 'psutil': True}


## 2. 从数据中生成结论草稿

先让程序提取可核对的事实，再由人写解释。程序可以告诉我们哪组 Recall 高、哪组 Chunk 数多，但不能替我们决定业务是否愿意用更多延迟换召回。

In [5]:
# 找出 top_k 实验中 Recall 最高的配置。
best_top_k = max(experiment_record["top_k_results"], key=lambda row: row["recall"])

# 找出 Chunk 数最少的配置，作为成本侧观察。
smallest_chunk_index = min(experiment_record["chunk_results"], key=lambda row: row["chunks"])

# 打印两个事实，后续报告正文会引用它们。
print("best top_k by recall:", best_top_k)
print("smallest index:", smallest_chunk_index)

# 检查自动提取的结果包含必要字段。
assert "recall" in best_top_k
assert "chunks" in smallest_chunk_index

best top_k by recall: {'top_k': 1, 'elapsed_ms': 0.03789999755099416, 'recall': 1.0, 'result_count': 1}
smallest index: {'chunk_size': 256, 'overlap': 64, 'chunks': 2, 'elapsed_ms': 0.0641999940853566, 'recall': 0.0}


## 3. 生成 Markdown 报告

报告使用当前运行真实数据填充，不手写一个可能与 Notebook 输出不一致的数字。`notes` 明确写出边界：样本小、没有并发、没有真实 ONNX 结果。

In [6]:
# 读取当前时间之外的稳定实验字段，避免报告每次无意义变化。
report_lines = [
    "# Phase 3 Baseline Report",
    "",
    "## 实验范围",
    "",
    "本报告来自本地小型 Markdown 语料，检索器为 BM25，目标是验证 benchmark 方法而不是宣称生产 SLA。",
    "",
    "## 计时基线",
    "",
    f"- index_version: `{timing_record['index_version']}`",
    f"- query: `{timing_record['query']}`",
    f"- warmup: `{timing_record['warmup']}`",
    f"- iterations: `{timing_record['iterations']}`",
    f"- mean_ms: `{timing_record['mean_ms']:.4f}`",
    f"- p50_ms: `{timing_record['p50_ms']:.4f}`",
    f"- p95_ms: `{timing_record['p95_ms']:.4f}`",
    "",
    "## 单变量实验",
    "",
    "### top_k",
]

# 把 top_k 实验逐行写入报告，保留完整条件。
for row in experiment_record["top_k_results"]:
    # 使用表格行记录 top_k、结果量、Recall 和耗时。
    report_lines.append(f"- top_k={row['top_k']}: results={row['result_count']}, recall={row['recall']:.3f}, elapsed_ms={row['elapsed_ms']:.4f}")

# 加入分块参数实验小节。
report_lines.extend(["", "### chunk_size/overlap", ""])

# 把分块实验逐行写入报告。
for row in experiment_record["chunk_results"]:
    # 记录参数、索引规模、Recall 和耗时。
    report_lines.append(f"- size={row['chunk_size']}, overlap={row['overlap']}: chunks={row['chunks']}, recall={row['recall']:.3f}, elapsed_ms={row['elapsed_ms']:.4f}")

# 加入优化边界，防止读者把 baseline 当成量化结论。
report_lines.extend(["", "## 限制与下一步", "", "- 当前语料规模很小，不能代表生产规模延迟。", "- 当前没有执行真实 ONNX/INT8 导出，因此不报告量化收益。", "- 下一步应固定更大数据集和 qrels，再比较 Dense/Hybrid 与 BM25。"])

# 将报告行连接成一个 Markdown 字符串。
report_text = "\n".join(report_lines) + "\n"

# 指定报告路径。
report_path = ROOT / "docs" / "phase3_baseline_report.md"

# 写入报告文件。
report_path.write_text(report_text, encoding="utf-8")

# 打印报告路径。
print("已生成:", report_path)

已生成: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\docs\phase3_baseline_report.md


## Phase 3 最终闸门

- [ ] 报告中的数字来自前面保存的实验记录。
- [ ] 同时报告质量、延迟、输入规模和实验条件。
- [ ] 明确写出当前没有执行的 ONNX/INT8 结论。
- [ ] 能解释为什么更快不自动等于更好。
- [ ] 已生成 `docs/phase3_baseline_report.md`。

## Boss Challenge：从报告中挑一条数字，补上数据版本、参数、环境和不能声称的内容。

下面是**故意保持注释状态**的跟敲模板。请先自己写，再取消注释逐行运行；不要把它当成需要复制的答案。

In [7]:
# 第 1 行：先写出本挑战需要的新变量或新输入。
# challenge_input = ...

# 第 2 行：调用本课已经学会的函数或模块。
# challenge_result = ...

# 第 3 行：打印一个中间结果，先观察再下结论。
# print(challenge_result)

# 第 4 行：写一个断言，把你的理解变成机器可检查的条件。
# assert ...

## 作品检查站

作品不是‘我运行过代码’，而是别人可以在文件浏览器中找到、下一阶段可以读取、你能解释生成过程的证据。下面的检查只报告事实，不替你假装通关。

In [8]:
# 列出本关应该产生的作品路径。
quest_artifact_candidates = ['docs/phase3_baseline_report.md']

# 把相对路径转换为项目根目录下的绝对路径。
quest_artifact_paths = [ROOT / path for path in quest_artifact_candidates]

# 只保留已经真正写入磁盘的作品。
quest_existing_artifacts = [str(path.relative_to(ROOT)) for path in quest_artifact_paths if path.is_file()]

# 保存一个不依赖外部服务的本关检查结果，方便复盘。
quest_checkpoint = {"stage": QUEST_STAGE, "existing_artifacts": quest_existing_artifacts}

# 打印检查结果，让学习者知道下一步是继续学习还是补交作品。
print("本关作品:", quest_existing_artifacts if quest_existing_artifacts else "还没有生成，请回到交付单元格")

本关作品: ['docs\\phase3_baseline_report.md']
